In [ ]:
import asyncio
import json

from dotenv import load_dotenv
from openai import AsyncOpenAI

from navgent.browser.browser_manager import BrowserManager
from navgent.browser.browser_session import BrowserSession
from navgent.tools import TOOL_SCHEMAS, TOOLS

MODEL = "gpt-4.1-mini"
MAX_STEPS = 3

SYSTEM_PROMPT = """
You are a browser agent. Use the tools to complete the task.
When the task is finished, reply with your answer instead of calling a tool.
"""


task = "Go to https://www.saucedemo.com/ and tell me what the login form asks for."
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": task},
]


async def run(task: str) -> str | None:
    load_dotenv()
    client = AsyncOpenAI()
    browser_manager = BrowserManager(headless=False)
    browser_session = BrowserSession(browser_manager)

    try:
        for _ in range(MAX_STEPS):
            response = await client.chat.completions.create(
                model=MODEL, messages=messages, tools=TOOL_SCHEMAS
            )
            message = response.choices[0].message
            messages.append(message)
            if not message.tool_calls:
                return message.content

            for call in message.tool_calls:
                print(call.function.name, call.function.arguments)
                tool = TOOLS[call.function.name]
                arguments = json.loads(call.function.arguments)
                output = await tool(browser_session, **arguments)
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": output,
                })
        return None
    finally:
        await browser_session.close()
        await browser_manager.stop()

In [ ]:
messages[3]

In [ ]:
await run(task)